# Duka Electronics Sales Analysis

## Engagement 1: January–June 2026

**Purpose:** Analyse Duka Electronics sales performance, identify the strongest branches and products, compare sales channels and days, and provide evidence-based recommendations for the Head of Sales.

## Executive Summary

- Duka Electronics recorded **1,000 orders** and generated total revenue of **KES 71,792,500** during January–June 2026.
- **Nairobi** was the highest-performing branch, generating **KES 26,106,100**, equivalent to **36.4%** of total revenue.
- **Laptops** were the leading product by revenue, contributing **KES 41,535,000**, while tablets led in units sold with **691 units**.
- **April** was the strongest month with **KES 13,518,300**, approximately **24.0%** higher than June, the weakest month.
- The business received **83 Premium Online orders** from Nairobi and Mombasa, generating **KES 10,218,000** in revenue.

In [ ]:
import pandas as pd
import numpy as np

from IPython.display import display

In [ ]:
sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

print("First five rows:")
display(sales.head())

print("Last five rows:")
display(sales.tail())

print("Dataset shape:")
print(sales.shape)

In [ ]:
print("DATA INFORMATION:")
sales.info()

print("\nDESCRIPTIVE STATISTICS:")
display(sales.describe())

print("\nDISTINCT VALUES PER COLUMN:")
display(sales.nunique())

print("\nMISSING VALUES:")
display(sales.isna().sum())

print("\nBRANCH DISTRIBUTION (%):")
display(
    (sales["branch"].value_counts(normalize=True) * 100).round(1)
)

print("\nCHANNEL DISTRIBUTION (%):")
display(
    (sales["channel"].value_counts(normalize=True) * 100).round(1)
)

print("\nEARLIEST ORDER:")
print(sales["order_date"].min())

print("\nLATEST ORDER:")
print(sales["order_date"].max())

## 1. First-Five-Minutes Inspection

- **Shape:** The dataset contains 1,000 rows and 7 original columns. Each row represents an order.
- **Data types:** `order_id` and `quantity` are integers, `order_date` is datetime, and the branch, product and channel columns contain categorical text. `unit_price` is numeric.
- **Missing values:** The inspection identifies no missing values in the generated dataset.
- **Date range:** Orders run from 1 January 2026 to 29 June 2026, covering the first half of 2026.
- **Key categories:** There are 5 branches, 5 products and 3 sales channels. Nairobi represents 37.6% of orders, while In-store represents 57.9% of orders.

The data is sufficiently complete for the planned sales analysis. The branch and channel distributions show that Nairobi and In-store sales account for the largest portions of the order volume.

In [ ]:
# Revenue per order
sales["revenue"] = (
    sales["quantity"] * sales["unit_price"]
)

# Month name
sales["month"] = sales["order_date"].dt.month_name()

# Weekday name
sales["weekday"] = sales["order_date"].dt.day_name()

# Weekend indicator
sales["is_weekend"] = (
    sales["order_date"].dt.dayofweek >= 5
)

# Premium or Value
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

print("Updated dataset:")
display(sales.head())

print("\nMONTH COUNTS:")
display(sales["month"].value_counts())

print("\nWEEKDAY COUNTS:")
display(sales["weekday"].value_counts())

print("\nWEEKEND COUNTS:")
display(sales["is_weekend"].value_counts())

print("\nPRICE BAND COUNTS:")
display(sales["price_band"].value_counts())

## 2. Feature Engineering

The revenue column represents the value of each order and is calculated by multiplying quantity by unit price. The month and weekday columns make it possible to compare sales over time. The `is_weekend` column distinguishes Saturday and Sunday orders from weekday orders. The `price_band` column classifies products costing KES 28,000 or more as Premium and the remaining products as Value.

In [ ]:
# Rank branches by revenue
branch_revenue = (
    sales.groupby("branch")
    .agg(
        total_revenue=("revenue", "sum"),
        number_of_orders=("order_id", "count")
    )
    .sort_values("total_revenue", ascending=False)
)

# Rank branches by number of orders
branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("BRANCH RANKING BY REVENUE:")
display(branch_revenue)

print("\nBRANCH RANKING BY NUMBER OF ORDERS:")
display(branch_orders)

## Task 3: Branch Performance

Nairobi ranks first in both revenue and number of orders, with KES 26,106,100 from 376 orders. Mombasa ranks second, followed by Nakuru, Kisumu and Eldoret.

The revenue and order rankings are the same in this dataset. This means that the branches with more orders also generate more total revenue in the same order. However, this does not necessarily mean that their average order values are identical.

In [ ]:
# Top three products by revenue
top_products_revenue = (
    sales.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

# Top three products by units sold
top_products_units = (
    sales.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("TOP THREE PRODUCTS BY REVENUE:")
display(top_products_revenue)

print("\nTOP THREE PRODUCTS BY UNITS SOLD:")
display(top_products_units)

## Task 4: Top Products

Laptops, tablets and phones are the top three products by revenue. Laptops lead with KES 41,535,000, followed by tablets with KES 19,348,000 and phones with KES 8,880,000.

By units sold, tablets lead with 691 units, followed by laptops with 639 units and phones with 592 units.

The lists differ because revenue depends on both quantity and selling price. Laptops generate the highest revenue despite selling fewer units than tablets because their unit price is much higher.

In [ ]:
weekend_comparison = (
    sales.groupby("is_weekend")
    .agg(
        number_of_orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        average_order_revenue=("revenue", "mean")
    )
)

weekend_comparison.index = [
    "Weekday" if not value else "Weekend"
    for value in weekend_comparison.index
]

print("WEEKDAY AND WEEKEND COMPARISON:")
display(weekend_comparison.round(2))

## Task 5: Weekend Versus Weekday Sales

There were 723 weekday orders generating KES 51,856,200, compared with 277 weekend orders generating KES 19,936,300.

Average order revenue was approximately KES 71,724 on weekdays and KES 71,972 on weekends. Weekend orders therefore had a slightly higher average order value, but weekdays generated substantially more total revenue because they had more orders.

The business should maintain weekday sales activity while exploring ways to increase weekend order volume.

In [ ]:
monthly_revenue = (
    sales.groupby(sales["order_date"].dt.to_period("M"))["revenue"]
    .sum()
    .sort_values(ascending=False)
)

best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("MONTHLY REVENUE:")
display(monthly_revenue)

print("\nBEST MONTH:", best_month)
print("BEST REVENUE:", best_revenue)

print("\nWORST MONTH:", worst_month)
print("WORST REVENUE:", worst_revenue)

print(
    "\nBest month is",
    round(percentage_difference, 2),
    "% higher than the worst month."
)

## Task 6: Best and Worst Month

April was the best month, generating KES 13,518,300. June was the worst month, generating KES 10,903,400.

April's revenue was approximately 24.0% higher than June's revenue, using June as the comparison base.

Management should investigate what contributed to April's stronger performance and whether similar conditions can be repeated in weaker months.

In [ ]:
premium_online_coast_capital = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("PREMIUM ONLINE ORDERS FROM NAIROBI AND MOMBASA:")
display(premium_online_coast_capital.head())

print(
    "\nNUMBER OF ORDERS:",
    len(premium_online_coast_capital)
)

print(
    "\nTOTAL REVENUE:",
    premium_online_coast_capital["revenue"].sum()
)

# Create output folder
import os
os.makedirs("outputs", exist_ok=True)

premium_online_coast_capital.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("\nCSV file exported successfully.")

## Task 8: Excel or Google Sheets Analysis

The spreadsheet analysis reproduces the branch revenue ranking using SUMIFS. An XLOOKUP formula adds the region associated with each branch. A PivotTable compares revenue by branch and sales channel.

The spreadsheet should produce the same branch revenue values and overall total as the pandas analysis. The overall revenue total is KES 71,792,500.

## Task 7: Premium Online Orders

There were 83 Premium Online orders from Nairobi and Mombasa. Their combined revenue was KES 10,218,000.

These orders were filtered using three conditions: the product price band had to be Premium, the channel had to be Online, and the branch had to be either Nairobi or Mombasa.

The filtered records were exported to `outputs/premium_online_coast_capital.csv` without an index column.

In [ ]:
# Calculate total revenue by branch
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Branch Revenue Ranking:")
display(branch_revenue)

print("Overall Revenue:")
print(sales["revenue"].sum())